# Solar Filament 2026 — Instance-PQ Championship Lab v4

**A legitimate, reproducible competition research system, not a precomputed submission.** It trains **YOLO26-Seg / YOLO11-Seg** on *competition-provided* masks, optionally trains a proposal-conditioned, stable crop refiner, compares single models/ensembles on grouped tuning observations, freezes one selection before an untouched audit, and creates `submission.csv` by running actual inference on the 180 hidden-label test images.

**Why v4:** The previous semantic ensemble scored **0.28–0.32 on public Kaggle**, and a competing claimed CPU 0.55 submission was generated from opaque precomputed RLE data, *not* its five-epoch U-Net. **No hardcoded test predictions, labels, private-test ground truth, or opaque payloads are used here.**

**This is a serious experiment, not an 0.80 promise.** Kaggle public/private scores differ; trust a held-out, multi-annotator, instance-PQ audit and real Kaggle submission. Follow challenge rules and licensing for pretrained weights (Ultralytics is AGPL-3.0). The organizer-style PQ logic should be compared with the official [self-evaluation notebook](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook) before claiming final parity.

**Run instructions:** Attach the *official competition data* with `Add Input`, enable T4/P100 GPU, enable Internet for initial Ultralytics installation and pretrained weight downloads (or attach licensed weights). For a full run, use `MODE='ALL'`. Save notebook output / model weights as a Kaggle dataset before the session ends. Later attach that output and set `MODE='SUBMIT'`. You may train one model per Kaggle session using `TRAIN_MODELS=['yolo26s-seg']` and then `['yolo11m-seg']`.

## 0. Configuration and dependencies
Modes: `SMOKE` (dataset and scorer smoke tests), `PREPARE`, `TRAIN`, `EVALUATE`, `SUBMIT`, `ALL`. `ALL` runs prepare → train → tune → audit → test. For incremental Kaggle sessions, attach your saved outputs and set `RESTORE_FROM` to the attached folder, or attach trained `best.pt` files via `MANUAL_CHECKPOINTS`. Training can be split across sessions.

In [ ]:
import os, sys, gc, json, time, random, math, shutil, hashlib, warnings, platform, subprocess, importlib.util
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, cv2, torch
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit

MODE = 'ALL'                      # 'SMOKE' | 'PREPARE' | 'TRAIN' | 'EVALUATE' | 'SUBMIT' | 'ALL'
SEED = 2026
ROOT = Path('/kaggle/working/solar_championship_v4') if Path('/kaggle/working').exists() else Path('/mnt/data/solar_championship_v4')
ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = None                  # leave None for autodiscovery
RESTORE_FROM = None              # optional /kaggle/input/your-saved-v4-output/solar_championship_v4
TRAIN_MODELS = ['yolo26s-seg', 'yolo11m-seg']  # sequential; change to one per Kaggle GPU session
# Point to your OWN trained or permitted checkpoints, e.g. {'yolo26s-seg':'/kaggle/input/.../best.pt'}
MANUAL_CHECKPOINTS = {}
# First run pretrained checkpoint download requires Internet. Or attach local pretrained weights.
PRETRAINED_OVERRIDES = {}        # e.g. {'yolo26s-seg':'/kaggle/input/weights/yolo26s-seg.pt'}
TRAIN_EPOCHS = {'yolo26s-seg':55, 'yolo11m-seg':45}
TRAIN_IMGSZ = {'yolo26s-seg':1024, 'yolo11m-seg':1024}
TRAIN_BATCH = {'yolo26s-seg':2, 'yolo11m-seg':2}
PREDICT_IMGSZ = 1536
RAW_CONF = 0.10                 # cache broad detector proposals; tune filters locally
RAW_IOU_NMS = 0.82
MAX_DET = 65
USE_HORIZONTAL_TTA = False       # optional: doubles inference cost and candidate volume
TRAIN_REFINER = True            # trained only on supplied allowed TRAIN annotations
REFINER_EPOCHS = 12
REFINER_SIDE = 320
REFINER_BATCH = 4
TUNE_MAX_IMAGES = 55            # choose a fixed tune subset, then test independently on ALL audit days
RUN_AUDIT_ONCE = True
TUNE_FAST = False               # set True only for smoke testing
USE_REFINER_IN_SEARCH = True
# Enables reference-algorithm implementation; not a guarantee of official scorer parity.
REQUIRE_NONOVERLAP = True
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
AMP = torch.cuda.is_available()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
cv2.setNumThreads(0)
print('Mode:',MODE,'CUDA:',torch.cuda.is_available(),'device:',torch.cuda.get_device_name(0) if AMP else 'CPU', 'root:',ROOT)
if MODE in ('ALL','TRAIN') and not AMP:
    raise RuntimeError('Training two instance models requires a GPU. Select GPU in Kaggle Notebook settings.')
for package, pipname in [('pycocotools','pycocotools'),('ultralytics','ultralytics>=8.3.0')]:
    if importlib.util.find_spec(package) is None:
        print('Installing missing package:', pipname)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pipname])
from pycocotools import mask as coco_mask
from ultralytics import YOLO
import ultralytics
print('Dependencies:',torch.__version__,ultralytics.__version__,cv2.__version__)

## 1. Competition data, provenance and grouped splits
A filename can appear in several different annotator records. All such records are kept in **one split**. Split by observation **day**, not by annotation ID. Models are trained only with permitted **training segmentation polygons**, never annotation spines, category names or test labels. We select the *medoid annotation set* for training when multiple people labeled the same physical image (rather than merging mutually incompatible polygons); tuning and audit score **all annotators separately**.

In [ ]:
def find_official_data():
    if DATA_ROOT is not None:
        paths=[Path(DATA_ROOT)]
    else:
        bases=[Path('/kaggle/input'),Path('/mnt/data')]
        paths=[p.parent.parent for base in bases if base.exists() for p in base.rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json')]
    for root in paths:
        if (root/'train'/'train_images').is_dir() and (root/'test'/'test_images').is_dir():
            return root
    raise FileNotFoundError('Attach the official MAGFiLO_1.0_Kaggle_2026 competition data in Kaggle Add Input.')
DATA=find_official_data()
annotations=json.loads((DATA/'train'/'MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
TRAIN_IMAGES={p.stem:p for p in (DATA/'train'/'train_images').iterdir() if p.suffix.lower() in ('.jpeg','.jpg','.png')}
TEST_IMAGES={p.stem:p for p in (DATA/'test'/'test_images').iterdir() if p.suffix.lower() in ('.jpeg','.jpg','.png')}
ann_map=defaultdict(list)
for a in annotations['annotations']: ann_map[str(a['image_id'])].append(a)
OBS=defaultdict(list)
for meta in annotations['images']:
    stem=Path(meta['file_name']).stem
    if stem in TRAIN_IMAGES:
        OBS[stem].append({'id':str(meta['id']),'anns':ann_map[str(meta['id'])]})
assert OBS and TEST_IMAGES and set(OBS).isdisjoint(TEST_IMAGES)
print('Unique training images:',len(OBS),'annotation versions:',sum(map(len,OBS.values())),'test images:',len(TEST_IMAGES))

def build_split():
    stems=sorted(OBS)
    groups=[x[:8] for x in stems]
    tr,others=next(GroupShuffleSplit(n_splits=1,test_size=.28,random_state=SEED).split(stems,groups=groups))
    other_stems=[stems[i] for i in others];other_groups=[groups[i] for i in others]
    tune, audit=next(GroupShuffleSplit(n_splits=1,test_size=.50,random_state=SEED+1).split(other_stems,groups=other_groups))
    return {'train':sorted(stems[i] for i in tr),'tune':sorted(other_stems[i] for i in tune),'audit':sorted(other_stems[i] for i in audit)}
SPLIT_PATH=ROOT/'split_manifest.json'
if SPLIT_PATH.exists():
    SPLIT=json.loads(SPLIT_PATH.read_text())
else:
    SPLIT=build_split(); SPLIT_PATH.write_text(json.dumps(SPLIT,indent=2))
for x in SPLIT:
    assert all(s in OBS for s in SPLIT[x]), (x,'missing data')
assert len(set().union(*[set(SPLIT[k]) for k in SPLIT]))==len(OBS)
assert not (set(SPLIT['train'])&set(SPLIT['tune']) or set(SPLIT['train'])&set(SPLIT['audit']) or set(SPLIT['tune'])&set(SPLIT['audit']))
for a in SPLIT:
    for b in SPLIT:
        if a!=b:assert not ({s[:8] for s in SPLIT[a]} & {s[:8] for s in SPLIT[b]})
print('Physical grouped split:',{key:len(v) for key,v in SPLIT.items()})

def restore_run():
    if not RESTORE_FROM:return
    source=Path(RESTORE_FROM)
    old=json.loads((source/'split_manifest.json').read_text())
    if old != SPLIT:raise RuntimeError('Saved checkpoint belongs to a DIFFERENT split. Stop to avoid leakage.')
    for relative in ('runs','refiner','cache','selection.json','tuning_results.csv'):
        src=source/relative; dest=ROOT/relative
        if src.is_dir():
            shutil.copytree(src,dest,dirs_exist_ok=True)
        elif src.is_file(): shutil.copy2(src,dest)
    print('Restored:',source)
restore_run()

## 2. Correct COCO RLE masks and medoid labels
Our evaluator counts **all IoU > 0.5 correspondences** per annotation view, matching a published organizer-style port. Earlier Hungarian one-to-one local scoring is not reused. We also report false positives, false negatives and SQ/RQ. Note: official source code may change; compare against the organizer notebook prior to final publication.

In [ ]:
H=W=2048

def polys_of(ann):
    seg=ann.get('segmentation',[])
    if isinstance(seg,dict): return []
    if seg and isinstance(seg[0],(int,float)): seg=[seg]
    paths=[]
    for pts in seg:
        if not isinstance(pts,list) or len(pts)<6 or len(pts)%2:continue
        arr=np.array(pts,dtype=np.float32).reshape(-1,2)
        if np.isfinite(arr).all(): paths.append(arr)
    return paths

def ann_rle(a):
    seg=a['segmentation']
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):return coco_mask.frPyObjects(seg,H,W)
        return {'counts':seg['counts'].encode('ascii') if isinstance(seg['counts'],str) else seg['counts'],'size':[H,W]}
    polys=[]
    for arr in polys_of(a):
        coords=np.clip(arr,[0,0],[W-1,H-1]).astype(np.float64).ravel().tolist()
        if len(coords)>=6: polys.append(coords)
    if not polys:return None
    result=coco_mask.frPyObjects(polys,H,W)
    return coco_mask.merge(result) if isinstance(result,list) else result

GT_CACHE={}
def gt_rles(stem):
    if stem not in GT_CACHE:
        GT_CACHE[stem]=[]
        for version in OBS[stem]:
            rles=[x for a in version['anns'] if (x:=ann_rle(a)) is not None]
            GT_CACHE[stem].append((version['id'],rles))
    return GT_CACHE[stem]

def medoid_record(stem):
    versions=OBS[stem]
    if len(versions)==1:return versions[0]
    rles=[rr for _,rr in gt_rles(stem)]
    scores=[]
    for j,x in enumerate(rles):
        values=[]
        for k,y in enumerate(rles):
            if j==k:continue
            if not x or not y:
                values.append(0.);continue
            iou=coco_mask.iou(x,y,[0]*len(y))
            matched=(iou>.5)
            # symmetric instance recognition F1, robust to different counts
            values.append(2*int(matched.sum())/max(1,len(x)+len(y)))
        scores.append(float(np.mean(values)))
    best=int(np.argmax(scores))
    return versions[best]
MEDOIDS={s:medoid_record(s) for s in SPLIT['train']}
print('Medoid chosen for',len(MEDOIDS),'physical training observations')

def compare_rle_views(gt, pred):
    if not gt and not pred:return 0.,0,0,0,0,0
    if not gt:return 0.,0,len(pred),0,0,0
    if not pred:return 0.,0,0,len(gt),0,0
    iou=np.asarray(coco_mask.iou(gt,pred,[0]*len(pred)),dtype=np.float64)
    hits=iou>0.5
    return (float(iou[hits].sum()),int(hits.sum()),int((hits.sum(axis=0)==0).sum()),
            int((hits.sum(axis=1)==0).sum()),int(((iou>0).sum(axis=1)>1).sum()),
            int(((iou>0).sum(axis=0)>1).sum()))

def score_submission_dict(predictions, stems):
    total=np.zeros(6,dtype=np.float64); per_view=[]
    for stem in stems:
        pred=predictions.get(stem,[])
        for imgid,truth in gt_rles(stem):
            metrics=compare_rle_views(truth,pred)
            total+=metrics
            si,tp,fp,fn,otm,mto=metrics
            per_view.append({'stem':stem,'annotator_image_id':imgid,'pq':si/max(1.,tp+.5*fp+.5*fn),
                             'tp':tp,'fp':fp,'fn':fn,'one_to_many':otm,'many_to_one':mto})
    si,tp,fp,fn,otm,mto=total
    den=tp+.5*fp+.5*fn
    summary={'pq_pooled':float(si/den) if den else 0.,'pq_mean_per_view':float(np.mean([v['pq'] for v in per_view])) if per_view else 0.,
             'sq':float(si/tp) if tp else 0.,'rq':float(tp/den) if den else 0.,
             'tp':int(tp),'fp':int(fp),'fn':int(fn),'one_to_many':int(otm),'many_to_one':int(mto),
             'annotation_views':len(per_view),'physical_images':len(stems)}
    return summary, pd.DataFrame(per_view)

# Genuine unit tests on synthetic RLE: perfect match, miss, duplicate, 2 views.
def mk_rle_from_rect(y0,x0,y1,x1):
    arr=np.zeros((H,W),np.uint8);arr[y0:y1,x0:x1]=1
    return coco_mask.encode(np.asfortranarray(arr))
a=mk_rle_from_rect(10,10,50,50)
b=mk_rle_from_rect(70,70,110,110)
assert compare_rle_views([a],[a])[:4]==(1.,1,0,0)
assert compare_rle_views([a],[])[:4]==(0.,0,0,1)
assert compare_rle_views([a],[b])[:4]==(0.,0,1,1)
assert compare_rle_views([a,b],[a,b])[:4]==(2.,2,0,0)
print('COCO compressed RLE / organizer-style multi-view PQ tests passed.')

## 3. Convert legal polygons to YOLO instance labels
Input H-alpha pixels stay **grayscale**; Ultralytics may replicate grayscale internally into three identical channels for compatibility with pretrained backbones. We do **not** use handcrafted false-color channels or any test labels. For a repeated physical image, use its permitted medoid annotation set. The YOLO validation directory is built only from the tune split; **the untouched audit is not used during training or hyperparameter selection**.

In [ ]:
import yaml
YOLO_DATA=ROOT/'yolo_data'
def write_yolo_label(stem,version,target):
    lines=[]
    for ann in version['anns']:
        for arr in polys_of(ann):
            if len(arr)<3:continue
            arr=np.clip(arr.astype(np.float64),[0,0],[W-1,H-1])/[W,H]
            if np.abs(cv2.contourArea(arr.astype(np.float32)))<1e-8:continue
            lines.append('0 '+' '.join(f'{value:.7f}' for value in arr.ravel()))
    target.write_text('\n'.join(lines)+'\n' if lines else '')

def link_image(source,dest):
    if dest.exists():return
    try: os.symlink(source,dest)
    except OSError:shutil.copy2(source,dest)

def prepare_yolo_data():
    for subset,key in [('train','train'),('val','tune')]:
        imgdir=YOLO_DATA/'images'/subset; lbldir=YOLO_DATA/'labels'/subset
        imgdir.mkdir(parents=True,exist_ok=True);lbldir.mkdir(parents=True,exist_ok=True)
        for stem in tqdm(SPLIT[key],desc='YOLO '+subset):
            path=TRAIN_IMAGES[stem]
            link_image(path,imgdir/path.name)
            if subset=='train': selected=MEDOIDS[stem]
            else: selected=medoid_record(stem)
            write_yolo_label(stem,selected,lbldir/(stem+'.txt'))
    manifest={'path':str(YOLO_DATA),'train':'images/train','val':'images/val',
              'names':{0:'filament'},'nc':1}
    (YOLO_DATA/'solar.yaml').write_text(yaml.safe_dump(manifest,sort_keys=False))
    return YOLO_DATA/'solar.yaml'
YAML_PATH=YOLO_DATA/'solar.yaml'
if MODE in ('PREPARE','TRAIN','ALL'):
    YAML_PATH=prepare_yolo_data()
print('YOLO dataset YAML:',YAML_PATH)

## 4. Train two complementary pretrained instance segmenters
Primary: **YOLO26s-Seg** (current architecture) and **YOLO11m-Seg** (architecturally different and larger). These are trained exclusively against permitted competition training annotations. Pretrained general-purpose weights may have their own license and original non-competition training sources; confirm those are allowed by current competition rules. We deliberately avoid training on public MAGFiLO test-label mirrors.

For T4/P100 training, run one model per Kaggle session if needed. `best.pt` and `last.pt` are saved under `ROOT/runs/<model>/weights/`. Subsequent sessions can attach saved outputs. If CUDA OOM, lower `TRAIN_BATCH[model]` to 1 or set `TRAIN_IMGSZ[model]` to 896. Never assume a model's COCO validation metric equals competition PQ.

In [ ]:
RUNS=ROOT/'runs';RUNS.mkdir(exist_ok=True)
def model_best_path(model_name):
    if model_name in MANUAL_CHECKPOINTS:
        p=Path(MANUAL_CHECKPOINTS[model_name]); assert p.exists(),p; return p
    p=RUNS/model_name/'weights'/'best.pt'
    return p if p.exists() else None

def pretrained_source(model_name):
    p=PRETRAINED_OVERRIDES.get(model_name)
    if p is not None:
        if not Path(p).exists():raise FileNotFoundError(p)
        return str(p)
    return model_name+'.pt'

def train_yolo(model_name):
    current=model_best_path(model_name)
    complete=RUNS/model_name/'completed.json'
    if current is not None and (complete.exists() or model_name in MANUAL_CHECKPOINTS):
        print('Reuse completed checkpoint:',current)
        return current
    last=RUNS/model_name/'weights'/'last.pt'
    resume=last.exists()
    weights=str(last) if resume else pretrained_source(model_name)
    print('Resuming' if resume else 'Training',model_name,'from',weights)
    try:model=YOLO(weights)
    except Exception as exc:
        raise RuntimeError(f'Cannot load {weights}; enable Kaggle Internet or attach licensed model weights. {exc}') from exc
    try:
        if resume:
            model.train(resume=True)
        else:
            model.train(data=str(YAML_PATH),epochs=TRAIN_EPOCHS[model_name],imgsz=TRAIN_IMGSZ[model_name],
                    batch=TRAIN_BATCH[model_name],device=DEVICE,workers=2,amp=AMP,
                    optimizer='AdamW',lr0=0.0003,lrf=0.05,weight_decay=0.0005,cos_lr=True,
                    seed=SEED,deterministic=True,patience=18,save=True,save_period=5,
                    degrees=180,translate=0.04,scale=0.18,shear=0.,perspective=0.,
                    flipud=0.5,fliplr=0.5,mosaic=0.0,mixup=0.,copy_paste=0.,
                    hsv_h=0.,hsv_s=0.,hsv_v=0.,
                    cache=False,plots=False,val=True,overlap_mask=False,
                    project=str(RUNS),name=model_name,exist_ok=True,verbose=False)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise RuntimeError('GPU OOM; reduce TRAIN_BATCH to 1 (or TRAIN_IMGSZ to 896) and resume.')
    path=RUNS/model_name/'weights'/'best.pt'
    if not path.exists():raise RuntimeError('Training ended but no best.pt was saved: '+str(path))
    complete.write_text(json.dumps({'model':model_name,'epochs_requested':TRAIN_EPOCHS[model_name],'best_path':str(path)}))
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return path

if MODE in ('TRAIN','ALL'):
    for model_name in TRAIN_MODELS:
        print('Saved weights:',train_yolo(model_name))
print('Available model checkpoints:',{m:str(model_best_path(m)) for m in TRAIN_MODELS})

## 5. Optional proposal-conditioned crop refiner (trained from scratch, stable)
This refiner is **conditioned on a proposed mask** (not just the H-alpha crop), learns synthetic proposal errors from the allowed GT masks, uses fp32 loss, gradient clipping and a strictly finite *best* checkpoint. During inference its output is constrained near the detector proposal and gated by topology/mask-area checks. It is **not automatically used**: tuning tests whether its refinement really improves PQ. The previous V2 crop refiner became NaN; this one refuses to save NaN weights.

In [ ]:
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

class DoubleConv(nn.Module):
    def __init__(self,c1,c2):
        super().__init__()
        self.seq=nn.Sequential(nn.Conv2d(c1,c2,3,padding=1,bias=False),nn.GroupNorm(8,c2),nn.SiLU(),
            nn.Conv2d(c2,c2,3,padding=1,bias=False),nn.GroupNorm(8,c2),nn.SiLU())
    def forward(self,x):return self.seq(x)
class ProposalUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=DoubleConv(3,32);self.e2=DoubleConv(32,64);self.e3=DoubleConv(64,128)
        self.mid=DoubleConv(128,192);self.d3=DoubleConv(192+128,128)
        self.d2=DoubleConv(128+64,64);self.d1=DoubleConv(64+32,32)
        self.head=nn.Conv2d(32,1,1)
    def forward(self,x):
        a=self.e1(x);b=self.e2(F.avg_pool2d(a,2));c=self.e3(F.avg_pool2d(b,2))
        d=self.mid(F.avg_pool2d(c,2))
        d=self.d3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode='bilinear',align_corners=False),c],dim=1))
        d=self.d2(torch.cat([F.interpolate(d,size=b.shape[-2:],mode='bilinear',align_corners=False),b],dim=1))
        d=self.d1(torch.cat([F.interpolate(d,size=a.shape[-2:],mode='bilinear',align_corners=False),a],dim=1))
        return self.head(d)

def crop_box_from_xyxy(box,image_shape=(H,W),context=1.65):
    x0,y0,x1,y1=map(float,box)
    side=max(x1-x0,y1-y0,32.)*context
    side=int(np.clip(math.ceil(side),min(80,min(image_shape)),min(image_shape)))
    xc=(x0+x1)/2;yc=(y0+y1)/2
    left=int(round(np.clip(xc-side/2,0,image_shape[1]-side)))
    top=int(round(np.clip(yc-side/2,0,image_shape[0]-side)))
    return left,top,side

def crop_patch(src,bounds,size=REFINER_SIDE,interpolation=cv2.INTER_LINEAR):
    x,y,s=bounds
    return cv2.resize(src[y:y+s,x:x+s],(size,size),interpolation=interpolation)

def refiner_channels(gray,proposal):
    base=np.asarray(gray,dtype=np.uint8)
    return np.stack([base.astype('float32')/255,
       cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(base).astype('float32')/255,
       proposal.astype('float32')],axis=0)

class TrainCrops(Dataset):
    def __init__(self,stems,augment=True,max_instances=10000):
        self.items=[];self.augment=augment
        for stem in stems:
            rec=MEDOIDS[stem]
            for ann in rec['anns']:
                if polys_of(ann):self.items.append((stem,ann))
        self.items=self.items[:max_instances]
        assert self.items
    def __len__(self):return len(self.items)
    def __getitem__(self,index):
        stem,ann=self.items[index]
        raw=cv2.imread(str(TRAIN_IMAGES[stem]),cv2.IMREAD_GRAYSCALE)
        p=polys_of(ann)[0]
        bbox=np.concatenate([p.min(0),p.max(0)])
        context=float(random.uniform(1.5,2.1)) if self.augment else 1.75
        bounds=crop_box_from_xyxy(bbox,raw.shape,context=context)
        patch=crop_patch(raw,bounds)
        x0,y0,side=bounds
        coords=np.rint((p-[x0,y0])*[REFINER_SIDE/side]*2).astype(np.int32)
        truth=np.zeros((REFINER_SIDE,REFINER_SIDE),np.uint8)
        cv2.fillPoly(truth,[coords],1)
        # Training proposal = imperfect detector-like dilation/erosion/translation.
        proposal=truth.copy()
        if self.augment:
            k=random.choice([1,3,5,7,9])
            kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k))
            if random.random()<.6:proposal=cv2.dilate(proposal,kernel)
            elif random.random()<.5:proposal=cv2.erode(proposal,kernel)
            dx,dy=random.randint(-4,4),random.randint(-4,4)
            M=np.float32([[1,0,dx],[0,1,dy]])
            proposal=cv2.warpAffine(proposal,M,(REFINER_SIDE,REFINER_SIDE))
            if random.random()<.5:
                patch=np.fliplr(patch);truth=np.fliplr(truth);proposal=np.fliplr(proposal)
            if random.random()<.5:
                patch=np.flipud(patch);truth=np.flipud(truth);proposal=np.flipud(proposal)
        if proposal.sum()==0: proposal=truth.copy()
        x=np.ascontiguousarray(refiner_channels(np.ascontiguousarray(patch),np.ascontiguousarray(proposal)))
        return torch.from_numpy(x),torch.from_numpy(np.ascontiguousarray(truth[None],dtype=np.float32))

def refiner_loss(logits,truth):
    logits=logits.float(); truth=truth.float()
    bce=F.binary_cross_entropy_with_logits(logits,truth,pos_weight=torch.tensor(4.,device=logits.device))
    prob=torch.sigmoid(logits)
    dice=1-(2*(prob*truth).sum((2,3))+1)/((prob+truth).sum((2,3))+1)
    return bce+.7*dice.mean()
REFINER_DIR=ROOT/'refiner';REFINER_DIR.mkdir(exist_ok=True)
REFINER_PATH=REFINER_DIR/'best_finite.pt'

def train_refiner():
    if REFINER_PATH.exists():
        print('Reusing stable crop refiner:',REFINER_PATH);return REFINER_PATH
    if not torch.cuda.is_available():raise RuntimeError('Enable a GPU before training the crop refiner')
    ds=TrainCrops(SPLIT['train'])
    loader=DataLoader(ds,batch_size=REFINER_BATCH,shuffle=True,num_workers=2,pin_memory=True)
    model=ProposalUNet().cuda()
    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=True)
    best=float('inf');history=[]
    for epoch in range(REFINER_EPOCHS):
        model.train();losses=[]
        for inp,gt in tqdm(loader,desc=f'Refiner {epoch+1}/{REFINER_EPOCHS}',leave=False):
            inp,gt=inp.cuda(non_blocking=True),gt.cuda(non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=True):logits=model(inp)
            loss=refiner_loss(logits,gt)
            if not torch.isfinite(loss):raise RuntimeError('Nonfinite refiner loss; saved best checkpoint preserved')
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.detach().cpu()))
        mean=float(np.mean(losses));history.append(mean)
        print(f'Refiner epoch {epoch+1}: {mean:.6f}')
        if np.isfinite(mean) and mean<best:
            state={k:v.cpu() for k,v in model.state_dict().items()}
            if any(not torch.isfinite(v).all() for v in state.values() if v.is_floating_point()):
                raise RuntimeError('Non-finite refiner model weights; prior best checkpoint preserved')
            best=mean
            torch.save({'model':state,'epoch':epoch+1,'train_loss':mean},REFINER_PATH)
    (REFINER_DIR/'losses.json').write_text(json.dumps(history))
    del model;torch.cuda.empty_cache();gc.collect()
    return REFINER_PATH

if MODE in ('ALL','TRAIN') and TRAIN_REFINER:train_refiner()

## 6. Live test-image inference and compact, provenance-linked proposal cache
This code calls `YOLO(best.pt).predict()` on **actual images**, converts predicted masks to compressed COCO RLE, and stores model/checkpoint metadata. No precomputed test predictions or hidden-label payloads. Each cached proposal is accompanied by the source model's SHA-256 and settings. Tuned thresholds may be applied to cached proposals without rerunning the network.

In [ ]:
HASH_MEMO={}
def sha256(path):
    path=Path(path);st=path.stat();key=(str(path.resolve()),st.st_size,st.st_mtime_ns)
    if key not in HASH_MEMO:
        h=hashlib.sha256()
        with open(path,'rb') as f:
            for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
        HASH_MEMO[key]=h.hexdigest()
    return HASH_MEMO[key]

def checkpoint_for(model_name):
    p=model_best_path(model_name)
    return Path(p) if p else None

def available_models():
    names=sorted(set(TRAIN_MODELS)|set(MANUAL_CHECKPOINTS))
    out={name:checkpoint_for(name) for name in names}
    return {name:path for name,path in out.items() if path and path.exists()}

def to_ascii_rle(mask):
    encoded=coco_mask.encode(np.asfortranarray(mask.astype(np.uint8)))
    return encoded['counts'].decode('ascii')
def to_coco_rle(counts):
    return {'size':[H,W],'counts':counts.encode('ascii') if isinstance(counts,str) else counts}

def result_to_proposals(result,invflip=None):
    if result.masks is None or result.boxes is None:return []
    scores=result.boxes.conf.detach().cpu().numpy()
    proposals=[]
    masks=result.masks.data
    if len(scores)!=len(masks):raise RuntimeError('Mask/score count mismatch')
    for i,score in enumerate(scores):
        mask=(masks[i].detach().cpu().numpy()>0.5).astype(np.uint8)
        if mask.shape!=(H,W):mask=cv2.resize(mask,(W,H),interpolation=cv2.INTER_NEAREST)
        if invflip=='h':mask=np.ascontiguousarray(mask[:,::-1])
        if invflip=='v':mask=np.ascontiguousarray(mask[::-1,:])
        if not mask.any():continue
        rle=to_ascii_rle(mask)
        proposals.append({'score':round(float(score),7),'counts':rle,'area':int(mask.sum())})
    return proposals

CACHE=ROOT/'cache';CACHE.mkdir(exist_ok=True)
def cache_path(model_name,stem):
    return CACHE/model_name/(stem+'.json')

def infer_one(model_name,model,stem,path,allow_tta):
    raw=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    if raw is None or raw.shape!=(H,W):raise RuntimeError(f'Bad image: {path}')
    proposals=[]
    views=[('original',raw,None)]
    if allow_tta:views.append(('horizontal',np.ascontiguousarray(raw[:,::-1]),'h'))
    for view_name,image,inv in views:
        # YOLO internally handles grayscale -> compatible 3-channel tensor; no color assumptions.
        # Ultralytics numpy source expects HWC BGR; all three channels are identical H-alpha intensity.
        detector_input=cv2.cvtColor(image,cv2.COLOR_GRAY2BGR)
        pred=model.predict(source=detector_input,imgsz=PREDICT_IMGSZ,conf=RAW_CONF,iou=RAW_IOU_NMS,
                   retina_masks=True,agnostic_nms=True,max_det=MAX_DET,
                   device=DEVICE,verbose=False,half=AMP)[0]
        proposals.extend(result_to_proposals(pred,invflip=inv))
    return proposals

def ensure_proposal_cache(model_name,stems,source_images,tta=USE_HORIZONTAL_TTA):
    ckpt=checkpoint_for(model_name)
    if ckpt is None:raise RuntimeError(f'Missing checkpoint for {model_name}; train or attach best.pt')
    digest=sha256(ckpt)
    folder=CACHE/model_name;folder.mkdir(parents=True,exist_ok=True)
    pending=[]
    for stem in stems:
        dest=cache_path(model_name,stem)
        if dest.exists():
            try:
                meta=json.loads(dest.read_text())
                if meta.get('checkpoint_sha256')==digest and meta.get('predict_imgsz')==PREDICT_IMGSZ and meta.get('tta')==tta:
                    continue
            except Exception:pass
        pending.append(stem)
    print(model_name,'cached',len(stems)-len(pending),'pending',len(pending))
    if pending:
        net=YOLO(str(ckpt))
        for stem in tqdm(pending,desc='Live '+model_name+' inference'):
            props=infer_one(model_name,net,stem,source_images[stem],tta)
            value={'stem':stem,'checkpoint_sha256':digest,'predict_imgsz':PREDICT_IMGSZ,
                   'tta':tta,'source':'LIVE_IMAGE_INFERENCE','proposals':props}
            cache_path(model_name,stem).write_text(json.dumps(value,separators=(',',':')))
        del net;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()

def read_pool(stem,names):
    pooled=[]
    for name in names:
        file=cache_path(name,stem)
        if not file.exists():raise FileNotFoundError(f'{file}; run EVALUATE before SUBMIT')
        value=json.loads(file.read_text())
        if value.get('source')!='LIVE_IMAGE_INFERENCE':raise RuntimeError('Reject opaque/test-target prediction payload')
        ckpt=checkpoint_for(name)
        if not ckpt or sha256(ckpt)!=value['checkpoint_sha256']:raise RuntimeError('Stale or mismatched cache: '+name)
        pooled.extend([{**p,'source_model':name} for p in value['proposals']])
    return pooled

## 7. Strict disjoint instance decoding + topology-aware proposal refinement
The competition is **instance segmentation**, not foreground Dice. Tune detector confidence, instance size, duplicate suppression and *whether* to use the refiner. For contested pixels, assign ownership in confidence order. Missing/tiny masks are removed. Overlapping output masks are **forbidden**.

In [ ]:
def load_refiner():
    if not REFINER_PATH.exists():return None
    ck=torch.load(REFINER_PATH,map_location='cpu',weights_only=False)
    state=ck['model']
    if any(not torch.isfinite(v).all() for v in state.values() if isinstance(v,torch.Tensor) and v.is_floating_point()):
        raise RuntimeError('Refiner has non-finite tensor: refusing to load')
    net=ProposalUNet();net.load_state_dict(state,strict=True)
    net=net.to('cuda' if torch.cuda.is_available() else 'cpu').eval()
    return net

@torch.inference_mode()
def refine_instance(net,gray,proposal):
    if net is None:return proposal
    ys,xs=np.nonzero(proposal)
    if len(ys)<50:return proposal
    bbox=[xs.min(),ys.min(),xs.max()+1,ys.max()+1]
    bounds=crop_box_from_xyxy(bbox,gray.shape,context=1.65)
    image_crop=crop_patch(gray,bounds)
    proposal_crop=crop_patch(proposal.astype('uint8'),bounds,interpolation=cv2.INTER_NEAREST)
    inp=torch.from_numpy(refiner_channels(image_crop,proposal_crop)[None]).to(next(net.parameters()).device)
    with torch.autocast('cuda',enabled=next(net.parameters()).device.type=='cuda'):
        prob=torch.sigmoid(net(inp))[0,0].float().cpu().numpy()
    x,y,s=bounds
    pp=cv2.resize(prob,(s,s),interpolation=cv2.INTER_LINEAR)>0.52
    # proposal-constrained topology guard: modest expansion only, no entirely new region
    allowed=cv2.dilate(proposal[y:y+s,x:x+s].astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9)))>0
    pp &= allowed
    proposed=np.zeros_like(proposal,dtype=bool);proposed[y:y+s,x:x+s]=pp
    ratio=proposed.sum()/max(1,proposal.sum())
    if ratio<.65 or ratio>1.55:return proposal
    union=np.logical_or(proposal,proposed).sum()
    agreement=np.logical_and(proposal,proposed).sum()/max(1,union)
    if agreement<.52:return proposal
    return proposed

def decode_and_postprocess(pool,params,gray=None,refiner=None):
    conf=float(params['conf']);area=int(params['min_area']);dup=float(params['duplicate_iou'])
    accepted=[];occupied=np.zeros((H,W),np.uint8)
    entries=sorted([p for p in pool if p['score']>=conf and p['area']>=area],
                   key=lambda p:(-p['score'],-p['area']))
    # pairwise near-duplicate rejection before exclusive allocation
    prev=[]
    for p in entries:
        rle=to_coco_rle(p['counts'])
        if prev:
            sim=coco_mask.iou([rle],prev,[0]*len(prev))[0]
            if len(sim) and sim.max()>=dup:continue
        prev.append(rle)
        mask=coco_mask.decode(rle).astype(bool)
        if params.get('refine',False) and refiner is not None:
            if gray is None:raise ValueError('Refinement requires the original image')
            mask=refine_instance(refiner,gray,mask)
        clean=mask & (~occupied.astype(bool))
        if clean.sum()<area:continue
        occupied[clean]=1
        accepted.append(coco_mask.encode(np.asfortranarray(clean.astype(np.uint8))))
    return accepted

def validate_disjoint(rles):
    # RLE intersection via COCO C-api; avoids building a full HxWxN tensor
    if len(rles)<2:return True
    pair=coco_mask.iou(rles,rles,[0]*len(rles))
    np.fill_diagonal(pair,0.)
    return bool(np.all(pair<=0.))

u=mk_rle_from_rect(5,5,45,45)
v=mk_rle_from_rect(20,20,70,70)
toy=[{'score':.9,'area':1600,'counts':u['counts'].decode('ascii')},
     {'score':.8,'area':2500,'counts':v['counts'].decode('ascii')}]
parameter_smoke={'conf':.2,'min_area':1,'duplicate_iou':.8,'refine':False}
toyout=decode_and_postprocess(toy,parameter_smoke)
assert len(toyout)==2 and validate_disjoint(toyout)
assert score_submission_dict({},[]) [0]['pq_pooled']==0.
print('NMS, COCO-RLE and disjoint instance tests passed')

## 8. Genuine model selection by held-out instance PQ
Evaluate on all annotators for each observation. **Tune** chooses a model and thresholds; **audit** is evaluated exactly once and never used for parameter search. We report both pooled and per-view PQ to make the aggregation assumption transparent. An ensemble is retained only if it beats its best single model on tune. Optionally compare refinement; do not force it if it hurts.

In [ ]:
from itertools import combinations,product

def tuning_stems():
    stems=sorted(SPLIT['tune']);rng=random.Random(SEED+77)
    rng.shuffle(stems)
    return sorted(stems[:min(len(stems),TUNE_MAX_IMAGES)])

def proposed_configs(names):
    sets=[(x,) for x in names]
    if len(names)>1: sets.append(tuple(names))
    confs=[.20,.32,.45] if not TUNE_FAST else [.28]
    areas=[140,350,700] if not TUNE_FAST else [350]
    dupes=[.50,.75] if not TUNE_FAST else [.65]
    ref_modes=[False,True] if USE_REFINER_IN_SEARCH and REFINER_PATH.exists() and not TUNE_FAST else [False]
    for group,conf,area,dup,ref in product(sets,confs,areas,dupes,ref_modes):
        yield {'models':list(group),'conf':conf,'min_area':area,'duplicate_iou':dup,'refine':ref}

def evaluate_candidate(params,stems,refiner=None):
    predictions={}
    for stem in stems:
        pool=read_pool(stem,params['models'])
        gray=cv2.imread(str(TRAIN_IMAGES[stem]),cv2.IMREAD_GRAYSCALE) if params['refine'] else None
        rles=decode_and_postprocess(pool,params,gray,refiner)
        if not validate_disjoint(rles):raise RuntimeError('Instance overlap detected; invalid candidate')
        predictions[stem]=rles
    summary,details=score_submission_dict(predictions,stems)
    return summary,details

SELECT=ROOT/'selection.json';TUNE_CSV=ROOT/'tuning_results.csv'
def tune_and_freeze():
    # Never re-select after an independent audit has been inspected.
    if SELECT.exists():
        saved=json.loads(SELECT.read_text())
        current={k:sha256(v) for k,v in available_models().items()}
        if current==saved.get('available_checkpoint_hashes'):
            print('Selection already frozen; reusing selected configuration.')
            return saved
        if (ROOT/'audit_report.json').exists():
            raise RuntimeError('Models changed after frozen audit. Start a NEW experiment folder instead of retrofitting the audit.')
        print('Checkpoint set changed, new selection needed; prior choice archived.')
        SELECT.replace(ROOT/'selection_previous.json')
    names=list(available_models())
    if not names:raise RuntimeError('No YOLO best.pt. Train / attach valid checkpoints first.')
    selected_stems=tuning_stems()
    for name in names:
        ensure_proposal_cache(name,selected_stems,TRAIN_IMAGES)
    refiner=load_refiner() if USE_REFINER_IN_SEARCH and REFINER_PATH.exists() else None
    configs=list(proposed_configs(names))
    results=[]
    for idx,params in enumerate(tqdm(configs,desc='Grouped tuning PQ')):
        try:
            score,_=evaluate_candidate(params,selected_stems,refiner)
            results.append({'index':idx,**params,**score})
        except Exception as e:
            # Fail loudly rather than blindly promote a possibly malformed submission
            print('Tuning configuration error:',params,repr(e));raise
    board=pd.DataFrame(results).sort_values(['pq_pooled','rq'],ascending=False)
    board.to_csv(TUNE_CSV,index=False)
    best=board.iloc[0]
    winner=configs[int(best['index'])]
    official_like_score=float(best['pq_pooled'])
    packaged={'params':winner,'tune_pq':official_like_score,'tune_pq_per_view':float(best['pq_mean_per_view']),
              'tune_images':selected_stems,'available_checkpoint_hashes':{k:sha256(v) for k,v in available_models().items()},
              'refiner_sha256':sha256(REFINER_PATH) if REFINER_PATH.exists() else None,
              'selection_method':'tune_only','seed':SEED}
    SELECT.write_text(json.dumps(packaged,indent=2))
    print('SELECTED PARAMS',json.dumps(packaged,indent=2))
    print('Top tuning configurations:');display(board.head(12))
    return packaged

if MODE in ('ALL','EVALUATE'):
    chosen=tune_and_freeze()
else:
    chosen=json.loads(SELECT.read_text()) if SELECT.exists() else None

## 9. One-shot independent audit, no tuning on audit
This is the best evidence that a change helped. The audit uses *every* annotation view for observations belonging to held-out days. It is entirely separate from YOLO train/val and the 55-image threshold-tuning set. Do not repeatedly revise thresholds based on audit numbers.

In [ ]:
AUDIT_JSON=ROOT/'audit_report.json';AUDIT_DETAIL=ROOT/'audit_per_annotator.csv'
def audit_frozen():
    if chosen is None:raise RuntimeError('Run EVALUATE before audit')
    if AUDIT_JSON.exists():
        print('Audit already exists; preserving one-shot seal:');print(AUDIT_JSON.read_text());return
    params=chosen['params']
    for name in params['models']:ensure_proposal_cache(name,SPLIT['audit'],TRAIN_IMAGES)
    refiner=load_refiner() if params['refine'] else None
    summary,details=evaluate_candidate(params,SPLIT['audit'],refiner)
    report={'phase':'FROZEN_UNTOUCHED_AUDIT','params':params,'metrics':summary,
            'model_hashes':{m:sha256(checkpoint_for(m)) for m in params['models']}}
    AUDIT_JSON.write_text(json.dumps(report,indent=2));details.to_csv(AUDIT_DETAIL,index=False)
    print(json.dumps(report,indent=2))
if MODE in ('ALL','EVALUATE') and RUN_AUDIT_ONCE:
    audit_frozen()

## 10. Produce the actual competition CSV from live inference
The only permissible test inputs are the official test **JPEGs**. `submission.csv` stores one COCO compressed RLE *counts* string per predicted filament (`2048×2048`). Instances within an image are mutually exclusive. This cell checks IDs, missing images, duplicate IDs, invalid RLE, empty masks, and overlapping instances. Images with no confident filaments can have **zero rows**; no dummy filler masks are generated.

In [ ]:
def export_submission():
    if not SELECT.exists():raise FileNotFoundError('selection.json missing; run EVALUATE / attach v4 outputs before SUBMIT')
    selected=json.loads(SELECT.read_text())
    params=selected['params']
    for name in params['models']:
        p=checkpoint_for(name)
        if p is None or sha256(p)!=selected['available_checkpoint_hashes'][name]:
            raise RuntimeError('Selection / checkpoint mismatch. Refuse to submit stale weights: '+name)
    if params['refine']:
        if not REFINER_PATH.exists() or sha256(REFINER_PATH)!=selected['refiner_sha256']:
            raise RuntimeError('Refiner checkpoint mismatch')
    for name in params['models']:
        ensure_proposal_cache(name,sorted(TEST_IMAGES),TEST_IMAGES)
    refiner=load_refiner() if params['refine'] else None
    rows=[];image_stats=[]
    for stem in tqdm(sorted(TEST_IMAGES),desc='Final legal test inference'):
        proposals=read_pool(stem,params['models'])
        gray=cv2.imread(str(TEST_IMAGES[stem]),cv2.IMREAD_GRAYSCALE) if params['refine'] else None
        rles=decode_and_postprocess(proposals,params,gray,refiner)
        if not validate_disjoint(rles):raise RuntimeError('Overlap found for '+stem)
        for ix,rle in enumerate(rles,1):
            if int(coco_mask.area(rle))<=0:raise RuntimeError('Empty instance')
            rows.append({'filament_id':f'{stem}_{ix}',
                         'segmentation_rle':rle['counts'].decode('ascii') if isinstance(rle['counts'],bytes) else rle['counts']})
        image_stats.append({'stem':stem,'raw':len(proposals),'final':len(rles)})
    frame=pd.DataFrame(rows,columns=['filament_id','segmentation_rle'])
    assert frame['filament_id'].is_unique and not frame.isna().any().any()
    output=Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else ROOT/'submission.csv'
    frame.to_csv(output,index=False)
    # Full round-trip RLE validation; no test ground truth accessed
    check=pd.read_csv(output,keep_default_na=False)
    for stem,indexed in check.groupby(check['filament_id'].str.rsplit('_',n=1).str[0]):
        if stem not in TEST_IMAGES:raise RuntimeError('Unexpected image ID '+stem)
        rles=[]
        for encoded in indexed.segmentation_rle:
            mask=to_coco_rle(encoded)
            if int(coco_mask.area(mask))==0:raise RuntimeError('Zero mask')
            rles.append(mask)
        if not validate_disjoint(rles):raise RuntimeError('Overlapping exported instances')
    pd.DataFrame(image_stats).to_csv(ROOT/'test_image_counts.csv',index=False)
    report={'submission':str(output),'rows':len(frame),'images_total':len(TEST_IMAGES),
            'images_with_instances':int((pd.DataFrame(image_stats)['final']>0).sum()),
            'params':params,'checkpoint_hashes':selected['available_checkpoint_hashes'],
            'sha256':sha256(output),'source':'LIVE_JPEG_INFERENCE',
            'uses_test_gt':False,'uses_precomputed_payload':False,
            'runtime':{'python':platform.python_version(),'torch':torch.__version__,
                       'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(0) if AMP else 'CPU'}}
    (ROOT/'submission_report.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2))
    return output
if MODE in ('ALL','SUBMIT'):
    path=export_submission()
    print('SUBMIT THIS FILE TO KAGGLE:',path)

## 11. Reproducibility and report export
Keep the source code, `split_manifest.json`, model weights, model SHA-256 values, `selection.json`, `audit_report.json`, `test_image_counts.csv` and exact dependency versions. The contest also requires a four-page technical report and a public repository with `requirements.txt` and a notebook that reproduces the solution. Include negative results and runtime, not only the best score.

In [ ]:
import importlib.metadata as pkgmeta
packages=['torch','torchvision','ultralytics','opencv-python','numpy','pandas','scipy','scikit-learn','pycocotools','PyYAML']
requirements=[]
for pkg in packages:
    try:requirements.append(pkg+'=='+pkgmeta.version(pkg))
    except pkgmeta.PackageNotFoundError:pass
(ROOT/'requirements_frozen.txt').write_text('\n'.join(requirements)+'\n')
manifest={
  'dataset':'MAGFiLO_1.0_Kaggle_2026 (official, no external test annotations)',
  'data_root':str(DATA), 'training_data_physical_images':len(OBS),
  'training_annotations':sum(len(v) for v in OBS.values()),
  'split_sizes':{k:len(v) for k,v in SPLIT.items()},
  'train_labels':'medoid set from competition-provided segmentation polygons',
  'selection':json.loads(SELECT.read_text()) if SELECT.exists() else None,
  'audit':json.loads(AUDIT_JSON.read_text()) if AUDIT_JSON.exists() else None,
  'not_an_official_score':'The final Kaggle public/private leaderboard score must be obtained by uploading the CSV',
  'sources':[
     'https://www.kaggle.com/competitions/filament-segmentation-2026/overview',
     'https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook',
     'https://docs.ultralytics.com/models/yolo26',
     'https://github.com/Wzy633/solar-filament-segmentation-2026',
     'https://github.com/QiuweiLiu/solar-filament-segmentation-2026'
  ]}
(ROOT/'research_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Reproducibility package at:',ROOT)
print('Next: upload /kaggle/working/submission.csv; compare Kaggle PQ, not training Dice.')